# POMDP agent training — parity/diff notebook for CPU, CuPy and S_v2 CUDA

This notebook is meant to replace the confusing all-states comparison notebook.

Goals:

1. Train CPU, CuPy and CUDA with the same explicit parameters.
2. Keep each backend in a separate result object and output directory.
3. Avoid reusing stale variables or overwriting DataFrames.
4. Generate the same policy-evaluation start points for all backends.
5. Produce tables that explain whether a backend did fewer iterations, fewer expansions, or simply reports different counters.
6. Compute semantic differences between learned value functions on a shared belief set when possible.

Important: on all-states environments with around 90k states, CUDA `v8` is currently the most viable S_v2 path. Avoid `auto_real` and `v7` for large multi-expansion all-states runs unless you are explicitly testing failure modes.

## 1. Imports and package path

Set `PACKAGE_ROOT` to the extracted S_v2 package. This cell clears old `olfnav_cuda_*` modules from the kernel so you do not accidentally mix versions.

In [ ]:
from pathlib import Path
import os
import sys
import json
import time
import traceback

import numpy as np
import pandas as pd
from matplotlib import pyplot as plt

# ---------------------------------------------------------------------
# Package auto-detection. Edit PACKAGE_ROOT manually if needed.
# ---------------------------------------------------------------------
PACKAGE_ROOT_CANDIDATES = [
    Path('/home/jlpfritas/HPC-POMDP/v1train_cuda/package_S_v2_pomdp_agent_training_style_allstates_fix3'),
    Path('/home/jlpfritas/HPC-POMDP/v3/package_S_v2_pomdp_agent_training_style_allstates_fix3'),
    Path('/home/jlpfritas/HPC-POMDP/v3/package_S_v2_final_real_cublas_traincuda_v1_clean'),
]
PACKAGE_ROOT = next((p for p in PACKAGE_ROOT_CANDIDATES if (p / 'python').exists()), PACKAGE_ROOT_CANDIDATES[0])
PACKAGE_PY = PACKAGE_ROOT / 'python'

if str(PACKAGE_PY) not in sys.path:
    sys.path.insert(0, str(PACKAGE_PY))

# Avoid stale package modules after switching notebook/package versions.
for name in list(sys.modules):
    if name.startswith('olfnav_cuda_backend') or name.startswith('olfnav_cuda_notebook'):
        del sys.modules[name]

from olfactory_navigation import Environment
from olfactory_navigation.agents import FSVI_Agent

from olfnav_cuda_notebook import (
    enable_cuda_backend,
    install_simulation_history_patch,
    normalize_environment_for_exact_converter,
    load_environment_from_metadata,
    read_environment_metadata,
    environment_kwargs_from_metadata,
    show_cuda_training_report,
    raw_start_points_from_environment,
    clean_start_points,
    generate_policy_start_points,
    run_policy_evaluation,
    run_policy_full_evaluation,
)

# Backend helper functions used only for analysis, not for training.
from olfnav_cuda_backend.fsvi_cuda_loop import vf_gamma, vf_actions, belief_matrix

install_simulation_history_patch(verbose=True)

CUDA_LIB = PACKAGE_ROOT / 'build' / 'libpomdp_backup_cuda.so'
print('Using PACKAGE_ROOT:', PACKAGE_ROOT)
print('CUDA_LIB:', CUDA_LIB)
print('CUDA_LIB exists:', CUDA_LIB.exists())

## 2. Global configuration

This notebook keeps the upstream `pomdp_agent_training` semantics: no `partitions`, no `minimal_converter`; `FSVI_Agent(env)` uses the full environment state space.

For all-states CUDA on large environments, start with `CUDA_VERSION='v8'`. If a CUDA OOM or illegal access happens, restart the kernel before the next CUDA test.

In [ ]:
# Environment generated from the reconstructed olfactory dataset.
# Change only this path when switching environment.
ENV_DIR = Path(
    '/home/jlpfritas/HPC-POMDP/v3/recon/generated_envs_large_then_downsample_thr3e6/envs/'
    'olfnav_F_large_t1750_len768_y40_250_x450_1225_thr3e-06_maxpool_target90000_states89946_from210x775_to131x486/'
    'Env-171_526-marg_20_20_20_20-edge_wrap_vertical-start_odor_present-source_85_465_radius2'
)

# Reproducibility / training parity.
SEED = 123
GAMMA = 0.95
EXPANSIONS = 100
MAX_BELIEF_GROWTH = 10
PRUNE_INTERVAL = 10
PRUNE_LEVEL = 1
UPDATE_PASSES = 1

# Native CPU/CuPy ag.train has an epsilon-based stopping criterion.
# Set to 0.0 to minimize early convergence stopping differences.
# If upstream dislikes 0.0, use 1e-15.
NATIVE_EPS = 0.0

# CUDA backend. For all-states large envs prefer v8.
CUDA_DEVICE = 1
CUDA_VERSION = 'v8'

# Run controls. For pure parity set all True; for debugging run one backend at a time.
RUN_BACKENDS = {
    'cpu': True,
    'cuda_v8': True,
    'cupy': True,
}
RUN_ORDER = ['cpu', 'cuda_v8', 'cupy']  # CUDA before CuPy keeps GPU memory cleaner.
STOP_ON_ERROR = True

# Policy evaluation.
RUN_POLICY_EVAL = True
RUN_FULL_POLICY_EVAL = False
N_EVAL = 100
EVAL_HORIZON = 1000
EVAL_USE_GPU = False  # intentional: compare trained policies, not simulator backend speed

# Output directory: separate from old notebooks.
OUT_ROOT = Path('tmp') / f'parity_diff_allstates_e{EXPANSIONS}_seed{SEED}'
OUT_ROOT.mkdir(parents=True, exist_ok=True)
print('OUT_ROOT:', OUT_ROOT)

## 3. Load and inspect the environment

The loader is metadata-driven, so source/margins/thresholds are not hard-coded. The normalization only converts list shapes to tuples and fixes non-layered metadata expected by upstream converters/plotters.

In [ ]:
def load_training_environment(env_dir: Path):
    return load_environment_from_metadata(
        env_dir,
        Environment,
        prefer_environment_load=True,
        verbose=True,
    )

try:
    env_meta = read_environment_metadata(ENV_DIR, verbose=True)
    env_kwargs_from_meta = environment_kwargs_from_metadata(ENV_DIR, env_meta, verbose=True)
except Exception as exc:
    print('Metadata audit skipped:', repr(exc))

env = load_training_environment(ENV_DIR)
normalize_environment_for_exact_converter(env, verbose=True)
print(env)
print('env.shape:', getattr(env, 'shape', None), type(getattr(env, 'shape', None)))
print('env.data_shape:', getattr(env, 'data_shape', None), type(getattr(env, 'data_shape', None)))
try:
    env.plot()
except Exception as exc:
    print('env.plot skipped:', repr(exc))

## 4. Agent factory

A fresh environment and a fresh agent are created for every backend. This avoids hidden cross-contamination between CPU, CuPy and CUDA.

In [ ]:
def make_env():
    e = load_training_environment(ENV_DIR)
    normalize_environment_for_exact_converter(e, verbose=False)
    return e


def make_agent(seed=SEED):
    e = make_env()
    print(f'[MAKE_AGENT] env_path={ENV_DIR}')
    print('[MAKE_AGENT] partitions=None / all environment states')
    print(f'[MAKE_AGENT] seed={seed}')
    ag = FSVI_Agent(
        e,
        seed=int(seed),
    )
    return ag

# Preview reference agent. This is not reused for training.
ag_preview = make_agent()
print('[PREVIEW] model_state_count=', getattr(ag_preview.model, 'state_count', None))
print('[PREVIEW] action_count=', getattr(ag_preview.model, 'action_count', None))
print('[PREVIEW] observation_count=', getattr(ag_preview.model, 'observation_count', None))

## 5. Utility functions for parity and diagnostics

These functions keep results in dictionaries, avoid variable-name confusion, and create comparable summaries.

In [ ]:
def native_if_cuda(agent):
    if agent is None:
        return None
    return getattr(agent, 'native_agent', agent)


def safe_json(x):
    try:
        json.dumps(x)
        return x
    except Exception:
        return str(x)


def history_summary_dict(hist):
    if hist is None:
        return {}
    s = getattr(hist, 'summary', None)
    if isinstance(s, dict):
        return dict(s)
    if s is not None:
        return {'summary_text': str(s)}
    return {'history_repr': str(hist)}


def infer_history_length(hist):
    if hist is None:
        return None
    # Try common attributes/methods without assuming upstream internals.
    for attr in ['history', 'rows', 'data', 'steps', 'times', 'timestamps', 'value_function_sizes', 'belief_set_sizes']:
        if hasattr(hist, attr):
            try:
                val = getattr(hist, attr)
                return len(val)
            except Exception:
                pass
    for meth in ['to_dataframe', 'as_dataframe']:
        if hasattr(hist, meth):
            try:
                return len(getattr(hist, meth)())
            except Exception:
                pass
    return None


def get_value_function(agent):
    agent = native_if_cuda(agent)
    if agent is None:
        return None
    return getattr(agent, 'value_function', None)


def get_belief_set(agent):
    agent = native_if_cuda(agent)
    if agent is None:
        return None
    for attr in ['belief_set', 'belief', 'beliefs']:
        if hasattr(agent, attr):
            val = getattr(agent, attr)
            if val is not None:
                return val
    return None


def count_alpha(agent):
    vf = get_value_function(agent)
    if vf is None:
        return None
    try:
        return int(len(vf))
    except Exception:
        pass
    try:
        return int(vf_gamma(vf).shape[0])
    except Exception:
        return None


def count_beliefs(agent):
    bs = get_belief_set(agent)
    if bs is None:
        return None
    try:
        return int(len(bs))
    except Exception:
        pass
    try:
        return int(belief_matrix(bs).shape[0])
    except Exception:
        return None


def train_record(label, kind, agent, result=None, history=None, df=None, error=None, t0=None, t1=None):
    native = native_if_cuda(agent)
    rec = {
        'label': label,
        'kind': kind,
        'agent': agent,
        'native_agent': native,
        'result': result,
        'history': history,
        'df': pd.DataFrame() if df is None else df,
        'error': error,
        'wall_s_outer': None if (t0 is None or t1 is None) else float(t1 - t0),
    }
    return rec


def summarize_train_records(records):
    rows = []
    for label, rec in records.items():
        result = rec.get('result')
        hist = rec.get('history')
        df = rec.get('df')
        error = rec.get('error')
        agent = rec.get('agent')

        if result is not None and hasattr(result, 'summary'):
            s = dict(result.summary)
            requested = s.get('requested_expansions')
            completed = s.get('completed_expansions')
            total_wall_s = s.get('total_wall_s')
            sum_backup_ms = s.get('sum_backup_ms')
            actual_versions = ','.join(sorted(set(map(str, df['actual_version'].dropna().unique())))) if df is not None and 'actual_version' in df.columns and len(df) else None
        else:
            s = history_summary_dict(hist)
            requested = EXPANSIONS
            completed = infer_history_length(hist)
            total_wall_s = rec.get('wall_s_outer')
            sum_backup_ms = None
            actual_versions = None

        rows.append({
            'label': label,
            'kind': rec.get('kind'),
            'status': 'ERROR' if error is not None else 'OK',
            'error': None if error is None else repr(error),
            'requested_expansions': requested,
            'completed_expansions_inferred': completed,
            'alpha_count': count_alpha(agent),
            'belief_count': count_beliefs(agent),
            'total_wall_s': total_wall_s,
            'outer_wall_s': rec.get('wall_s_outer'),
            'sum_backup_ms': sum_backup_ms,
            'actual_versions': actual_versions,
            'history_summary': safe_json(s),
        })
    return pd.DataFrame(rows)


def maybe_free_cupy_memory():
    try:
        import cupy as cp
        cp.cuda.Device(CUDA_DEVICE).use()
        cp.get_default_memory_pool().free_all_blocks()
        cp.get_default_pinned_memory_pool().free_all_blocks()
        print('[CuPy] freed default memory pools')
    except Exception as exc:
        print('[CuPy] memory pool cleanup skipped:', repr(exc))

## 6. Backend-specific training helpers

CPU/CuPy call the official upstream `ag.train(...)` with explicit `eps`. CUDA calls `traincuda(...)` with the same FSVI parameters where supported. CUDA does not implement upstream `eps` stopping; it attempts exactly `EXPANSIONS` iterations unless an error occurs.

In [ ]:
def train_cpu_backend(label='cpu'):
    ag = make_agent()
    t0 = time.perf_counter()
    hist = ag.train(
        expansions=EXPANSIONS,
        update_passes=UPDATE_PASSES,
        max_belief_growth=MAX_BELIEF_GROWTH,
        prune_interval=PRUNE_INTERVAL,
        prune_level=PRUNE_LEVEL,
        gamma=GAMMA,
        eps=NATIVE_EPS,
        use_gpu=False,
        overwrite_training=True,
        print_progress=True,
        print_stats=True,
    )
    t1 = time.perf_counter()
    print(hist.summary if hasattr(hist, 'summary') else hist)
    return train_record(label, 'cpu_native', ag, history=hist, t0=t0, t1=t1)


def train_cupy_backend(label='cupy'):
    maybe_free_cupy_memory()
    ag = make_agent()
    t0 = time.perf_counter()
    hist = ag.train(
        expansions=EXPANSIONS,
        update_passes=UPDATE_PASSES,
        max_belief_growth=MAX_BELIEF_GROWTH,
        prune_interval=PRUNE_INTERVAL,
        prune_level=PRUNE_LEVEL,
        gamma=GAMMA,
        eps=NATIVE_EPS,
        use_gpu=True,
        overwrite_training=True,
        print_progress=True,
        print_stats=True,
    )
    t1 = time.perf_counter()
    print(hist.summary if hasattr(hist, 'summary') else hist)
    return train_record(label, 'cupy_native_gpu', ag, history=hist, t0=t0, t1=t1)


def train_cuda_backend(label='cuda_v8', version=CUDA_VERSION):
    if not CUDA_LIB.exists():
        raise FileNotFoundError(f'Missing CUDA_LIB={CUDA_LIB}. Rebuild with scripts/31_build_backend_lib.sh')

    # For large all-states runs, avoid CuPy leftovers before CUDA.
    maybe_free_cupy_memory()

    ag_base = make_agent()
    ag_cuda = enable_cuda_backend(
        ag_base,
        device=CUDA_DEVICE,
        version=version,
        gamma=GAMMA,
        lib_path=CUDA_LIB,
    )
    print(f'[{label}] traincuda available:', hasattr(ag_cuda, 'traincuda'))
    print(f'[{label}] CUDA config:', getattr(ag_cuda, '_cuda_backend_config', {'version': version, 'device': CUDA_DEVICE, 'gamma': GAMMA}))

    t0 = time.perf_counter()
    res = ag_cuda.traincuda(
        expansions=EXPANSIONS,
        use_gpu=True,
        gamma=GAMMA,
        max_belief_growth=MAX_BELIEF_GROWTH,
        prune_interval=PRUNE_INTERVAL,
        prune_level=PRUNE_LEVEL,
        outdir=str(OUT_ROOT / f'train_{label}_e{EXPANSIONS}'),
        checkpoint_every=max(1, EXPANSIONS // 4),
        visual=True,
        display_rows=12,
    )
    t1 = time.perf_counter()
    df = pd.DataFrame(res.rows)
    print(json.dumps(res.summary, indent=2, default=str))
    return train_record(label, f'cuda_{version}', ag_cuda, result=res, df=df, t0=t0, t1=t1)

## 7. Execute selected training backends

Each backend writes to a distinct output folder and keeps its own variables inside `train_records`.

If CUDA fails with OOM or illegal memory access, restart the kernel before running another CUDA cell. The notebook can store the error, but the CUDA context may be dirty afterwards.

In [ ]:
train_records = {}

for label in RUN_ORDER:
    if not RUN_BACKENDS.get(label, False):
        print(f'[{label}] skipped by RUN_BACKENDS')
        continue

    print('\n' + '=' * 100)
    print('TRAINING BACKEND:', label)
    print('=' * 100)

    try:
        if label == 'cpu':
            rec = train_cpu_backend(label)
        elif label == 'cupy':
            rec = train_cupy_backend(label)
        elif label == 'cuda_v8':
            rec = train_cuda_backend(label, version=CUDA_VERSION)
        else:
            raise ValueError(f'Unknown backend label: {label}')
        train_records[label] = rec
    except Exception as exc:
        traceback.print_exc()
        train_records[label] = train_record(label, 'unknown', None, error=exc)
        if STOP_ON_ERROR:
            raise

print('\nFinished selected training backends:', list(train_records))

## 8. Training audit table

Use this table to check whether a backend really completed fewer iterations/expansions or whether the apparent difference came from different counters.

In [ ]:
df_train_summary = summarize_train_records(train_records)
display(df_train_summary)

df_train_summary.to_csv(OUT_ROOT / 'training_parity_summary.csv', index=False)
print('Saved:', OUT_ROOT / 'training_parity_summary.csv')

# Optional: show CUDA rows if available.
for label, rec in train_records.items():
    df = rec.get('df')
    if df is not None and len(df):
        print('\nCUDA/detail rows for', label)
        cols = [c for c in [
            'iter', 'nB', 'nG_in', 'actual_version', 'expand_ms', 'backup_ms',
            'backup_wall_ms', 'update_ms', 'iter_total_ms', 'alpha_after',
            'belief_total_after', 'pruned'
        ] if c in df.columns]
        display(df[cols].tail(20))
        df.to_csv(OUT_ROOT / f'{label}_training_rows.csv', index=False)

## 9. Generate shared policy-evaluation start points

Start points are generated from the loaded environment, not from a stale backend-specific agent. This makes CPU/CuPy/CUDA evaluation use the same starts.

In [ ]:
start_point_sets = generate_policy_start_points(
    env,
    n_eval=N_EVAL,
    out_root=OUT_ROOT,
    remove_source_points=True,
    verbose=True,
)
start_points_raw = start_point_sets['raw']
start_points_full = start_point_sets['full']
start_points_eval = start_point_sets['eval']

print('[START_POINTS] raw:', start_points_raw.shape)
print('[START_POINTS] full:', start_points_full.shape)
print('[START_POINTS] eval:', start_points_eval.shape)

## 10. Policy evaluation on identical starts

Evaluation uses `use_gpu=False` for all trained policies. This compares the policies learned by CPU/CuPy/CUDA without adding simulator backend differences.

In [ ]:
def print_hist_summary(label, hist):
    print('\n' + '=' * 100)
    print(label)
    print('=' * 100)
    if hist is None:
        print('None')
    elif hasattr(hist, 'summary'):
        print(hist.summary)
    else:
        print(hist)


def run_eval_for_record(label, rec, start_points, full=False):
    agent = rec.get('agent')
    if agent is None or rec.get('error') is not None:
        print(f'[{label}] evaluation skipped: no trained agent')
        return None
    hist = run_policy_evaluation(
        native_if_cuda(agent),
        start_points=start_points,
        n=len(start_points),
        horizon=EVAL_HORIZON,
        reward_discount=GAMMA,
        use_gpu=EVAL_USE_GPU,
        time_shift=False,
        time_loop=False,
        print_progress=True,
        print_stats=True,
    )
    print_hist_summary(f'{label} {"FULL" if full else "QUICK"} policy evaluation', hist)
    return hist

hist_eval_records = {}
if RUN_POLICY_EVAL:
    for label, rec in train_records.items():
        hist_eval_records[label] = run_eval_for_record(label, rec, start_points_eval, full=False)
else:
    print('Quick policy evaluation disabled')

hist_full_records = {}
if RUN_FULL_POLICY_EVAL:
    for label, rec in train_records.items():
        hist_full_records[label] = run_eval_for_record(label, rec, start_points_full, full=True)
else:
    print('Full policy evaluation disabled')

## 11. Evaluation summary table

This table lets you compare simulation outcomes on the same start set.

In [ ]:
def summarize_hist_records(hist_records, tag):
    rows = []
    for label, hist in hist_records.items():
        sd = history_summary_dict(hist)
        row = {'label': label, 'eval_type': tag, 'hist_type': type(hist).__name__ if hist is not None else None}
        # Keep both parsed common fields and raw summary text/dict.
        if isinstance(sd, dict):
            for k, v in sd.items():
                if k in ['summary_text', 'history_repr']:
                    row[k] = v
                elif isinstance(v, (int, float, str, bool, type(None))):
                    row[k] = v
            row['summary_raw'] = safe_json(sd)
        rows.append(row)
    return pd.DataFrame(rows)

df_eval_summary = summarize_hist_records(hist_eval_records, 'quick') if hist_eval_records else pd.DataFrame()
df_full_summary = summarize_hist_records(hist_full_records, 'full') if hist_full_records else pd.DataFrame()

display(df_eval_summary)
if len(df_full_summary):
    display(df_full_summary)

df_eval_summary.to_csv(OUT_ROOT / 'policy_eval_quick_summary.csv', index=False)
if len(df_full_summary):
    df_full_summary.to_csv(OUT_ROOT / 'policy_eval_full_summary.csv', index=False)

## 12. Value-function semantic diff on shared belief sets

Raw alpha-vector counts can differ even when policies are semantically close. This cell compares value functions by evaluating them on a reference belief matrix.

Recommended references:

- `reference_label='cuda_v8'`: compare all value functions on CUDA's belief set.
- `reference_label='cpu'`: compare all value functions on CPU's belief set.

If a backend did not run or does not expose a belief set/value function, it is skipped.

In [ ]:
def get_gamma_actions(agent):
    vf = get_value_function(agent)
    if vf is None:
        return None, None
    try:
        Gamma = vf_gamma(vf)
    except Exception as exc:
        print('vf_gamma failed:', repr(exc))
        return None, None
    try:
        actions = vf_actions(vf)
    except Exception:
        actions = None
    return Gamma, actions


def get_belief_matrix(agent):
    bs = get_belief_set(agent)
    if bs is None:
        return None
    try:
        return belief_matrix(bs)
    except Exception as exc:
        print('belief_matrix failed:', repr(exc))
        return None


def value_policy_on_B(B, Gamma, actions=None):
    scores = np.asarray(B, dtype=np.float64) @ np.asarray(Gamma, dtype=np.float64).T
    gstar = np.argmax(scores, axis=1).astype(np.int64)
    values = scores[np.arange(scores.shape[0]), gstar]
    selected_actions = None
    if actions is not None:
        arr = np.asarray(actions).ravel()
        if len(arr) > int(gstar.max(initial=0)):
            selected_actions = arr[gstar]
    return values, gstar, selected_actions


def compare_pair_on_B(label_a, agent_a, label_b, agent_b, B):
    Gamma_a, actions_a = get_gamma_actions(agent_a)
    Gamma_b, actions_b = get_gamma_actions(agent_b)
    if Gamma_a is None or Gamma_b is None or B is None:
        return {
            'pair': f'{label_a} vs {label_b}',
            'comparable': False,
            'reason': 'missing Gamma or B',
        }
    va, ga, aa = value_policy_on_B(B, Gamma_a, actions_a)
    vb, gb, ab = value_policy_on_B(B, Gamma_b, actions_b)
    diff = np.abs(va - vb)
    row = {
        'pair': f'{label_a} vs {label_b}',
        'comparable': True,
        'n_beliefs_ref': int(B.shape[0]),
        'nS': int(B.shape[1]),
        f'{label_a}_nG': int(Gamma_a.shape[0]),
        f'{label_b}_nG': int(Gamma_b.shape[0]),
        'max_abs_value_diff': float(diff.max(initial=0.0)),
        'mean_abs_value_diff': float(diff.mean()) if diff.size else 0.0,
        'median_abs_value_diff': float(np.median(diff)) if diff.size else 0.0,
        'value_diff_gt_1e-8': int((diff > 1e-8).sum()),
        'value_diff_gt_1e-6': int((diff > 1e-6).sum()),
    }
    if aa is not None and ab is not None and len(aa) == len(ab):
        mism = np.asarray(aa) != np.asarray(ab)
        row['action_mismatch_count'] = int(mism.sum())
        row['action_mismatch_rate'] = float(mism.mean()) if mism.size else 0.0
    else:
        row['action_mismatch_count'] = None
        row['action_mismatch_rate'] = None
    return row


def semantic_diff_table(reference_label=None):
    available = {label: rec['agent'] for label, rec in train_records.items() if rec.get('agent') is not None and rec.get('error') is None}
    if not available:
        print('No available trained agents')
        return pd.DataFrame()
    if reference_label is None:
        reference_label = next(iter(available))
    if reference_label not in available:
        raise ValueError(f'reference_label={reference_label!r} not in available labels: {list(available)}')

    B_ref = get_belief_matrix(available[reference_label])
    if B_ref is None:
        print('Reference belief matrix unavailable:', reference_label)
        return pd.DataFrame()
    print(f'[SEMANTIC_DIFF] Reference B from {reference_label}:', B_ref.shape)

    rows = []
    labels = list(available.keys())
    for i in range(len(labels)):
        for j in range(i + 1, len(labels)):
            rows.append(compare_pair_on_B(labels[i], available[labels[i]], labels[j], available[labels[j]], B_ref))
    return pd.DataFrame(rows)

# Choose a reference backend that actually ran.
preferred_refs = ['cuda_v8', 'cpu', 'cupy']
reference_label = next((x for x in preferred_refs if x in train_records and train_records[x].get('agent') is not None), None)
df_semantic_diff = semantic_diff_table(reference_label=reference_label)
display(df_semantic_diff)
if len(df_semantic_diff):
    df_semantic_diff.to_csv(OUT_ROOT / f'semantic_value_diff_ref_{reference_label}.csv', index=False)

## 13. Plot histories

These are native `SimulationHistory.plot()` calls. The package-level metadata patch should avoid the previous non-layered plotting error.

In [ ]:
PLOT_QUICK_HISTS = True
PLOT_FULL_HISTS = False

if PLOT_QUICK_HISTS:
    for label, hist in hist_eval_records.items():
        if hist is not None:
            print(f'Plot quick hist: {label}')
            hist.plot()

if PLOT_FULL_HISTS:
    for label, hist in hist_full_records.items():
        if hist is not None:
            print(f'Plot full hist: {label}')
            hist.plot()

## 14. Save final metadata

The metadata records requested parameters and the actual backend outputs. This avoids ambiguity from global variables being changed mid-notebook.

In [ ]:
metadata = {
    'package_root': str(PACKAGE_ROOT),
    'cuda_lib': str(CUDA_LIB),
    'env_dir': str(ENV_DIR),
    'state_space': 'all_environment_states',
    'seed': SEED,
    'gamma': GAMMA,
    'expansions': EXPANSIONS,
    'max_belief_growth': MAX_BELIEF_GROWTH,
    'prune_interval': PRUNE_INTERVAL,
    'prune_level': PRUNE_LEVEL,
    'update_passes': UPDATE_PASSES,
    'native_eps': NATIVE_EPS,
    'cuda_device': CUDA_DEVICE,
    'cuda_version': CUDA_VERSION,
    'run_backends': RUN_BACKENDS,
    'run_order': RUN_ORDER,
    'n_eval': None if N_EVAL is None else int(N_EVAL),
    'n_start_points_raw': int(len(start_points_raw)) if 'start_points_raw' in globals() else None,
    'n_start_points_full': int(len(start_points_full)) if 'start_points_full' in globals() else None,
    'n_start_points_eval': int(len(start_points_eval)) if 'start_points_eval' in globals() else None,
    'run_full_policy_eval': bool(RUN_FULL_POLICY_EVAL),
    'training_summary_csv': str(OUT_ROOT / 'training_parity_summary.csv'),
    'eval_summary_csv': str(OUT_ROOT / 'policy_eval_quick_summary.csv'),
}
(OUT_ROOT / 'notebook_run_metadata.json').write_text(json.dumps(metadata, indent=2, sort_keys=True, default=str) + '\n')
print('Saved metadata:', OUT_ROOT / 'notebook_run_metadata.json')